# Find Optimal Steering Layers

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/modulabs-personalab/psyctl/blob/main/examples/en/05_layer_analysis.ipynb)

Not all layers are equally good for steering. PSYCTL includes a layer analysis tool that uses SVM-based separation analysis to rank which layers best distinguish between "positive" (personality-exhibiting) and "neutral" activations.

**In this notebook you will:**
1. Analyze all MLP layers using a wildcard pattern
2. View the ranking table (score, accuracy, margin)
3. Visualize layer separation scores
4. Extract a vector from the top-ranked layer

**Requirements:** Colab free tier (T4 GPU). [HuggingFace token](https://huggingface.co/settings/tokens).

**Time:** ~10 minutes

In [ ]:
!pip install -q git+https://github.com/modulabs-personalab/psyctl.git bitsandbytes accelerate datasets

In [ ]:
import os

try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("Loaded HF_TOKEN from Colab Secrets")
except Exception:
    pass

if not os.environ.get("HF_TOKEN"):
    try:
        import getpass
        os.environ["HF_TOKEN"] = getpass.getpass("Enter your HuggingFace token: ")
    except Exception:
        raise RuntimeError(
            "HF_TOKEN not found. Add it to Colab Secrets (key icon in left sidebar) "
            "and enable notebook access, then re-run this cell."
        )

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PSYCTL_LOG_LEVEL"] = "WARNING"

## Prepare Dataset

We need a steering dataset to analyze. We'll download a community dataset from HuggingFace.

In [ ]:
from pathlib import Path
from datasets import load_dataset

ds = load_dataset("CaveduckAI/steer-personality-extroversion-ko", split="train")
dataset_dir = Path("./dataset_demo")
dataset_dir.mkdir(exist_ok=True)
dataset_path = dataset_dir / "data.jsonl"
ds.to_json(str(dataset_path))
print(f"Dataset: {len(ds)} samples saved to {dataset_path}")

## Layer Analysis Theory

When we feed "positive" and "neutral" texts through a model, their internal activations differ at each layer. Some layers show a clear separation between the two groups (good for steering), while others show activations that are mixed together (poor for steering).

The **SVM analyzer** trains a linear SVM classifier at each layer to separate positive from neutral activations, then reports:
- **Score**: Overall separation quality (higher is better)
- **Accuracy**: Classification accuracy (how well the SVM separates the two groups)
- **Margin**: SVM margin (larger margin = more robust separation)

## Analyze All MLP Layers

We use the wildcard pattern `model.layers[*].mlp` to analyze every MLP layer in the model.

In [ ]:
from psyctl.core.layer_analyzer import LayerAnalyzer

MODEL_NAME = "google/gemma-3-270m-it"
LAYER_PATTERNS = ["model.layers[*].mlp"]

analyzer = LayerAnalyzer()

print(f"Analyzing all MLP layers of {MODEL_NAME}...")
results = analyzer.analyze_layers(
    model_name=MODEL_NAME,
    layers=LAYER_PATTERNS,
    dataset_path=dataset_path,
    method="svm",
    top_k=5,
    batch_size=8,
)

print(f"\nAnalyzed {results['total_layers']} layers")
print(f"\nTop 5 Layers:")
print(f"{'Rank':<6} {'Layer':<40} {'Score':>8} {'Accuracy':>10} {'Margin':>8}")
print("-" * 74)
for i, r in enumerate(results["rankings"][:5], 1):
    m = r["metrics"]
    print(f"{i:<6} {r['layer']:<40} {m.get('score', 0):>8.4f} {m.get('accuracy', 0):>10.4f} {m.get('margin', 0):>8.4f}")

## Visualization

In [ ]:
import matplotlib.pyplot as plt
import re

layers = []
scores = []
for r in results["rankings"]:
    layer_name = r["layer"]
    # Extract layer number for cleaner labels
    match = re.search(r"\[(\d+)\]", layer_name)
    label = f"Layer {match.group(1)}" if match else layer_name
    layers.append(label)
    scores.append(r["metrics"].get("score", 0))

# Highlight top 5
colors = ["#E85D75" if i < 5 else "#4A90D9" for i in range(len(layers))]

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(range(len(layers)), scores, color=colors)
ax.set_xticks(range(len(layers)))
ax.set_xticklabels(layers, rotation=45, ha="right")
ax.set_ylabel("Separation Score")
ax.set_title(f"Layer Separation Scores — {MODEL_NAME}")
ax.axhline(y=scores[0] * 0.9, color="grey", linestyle="--", alpha=0.5, label="90% of best")
ax.legend()
plt.tight_layout()
plt.show()

## Extract from the Top Layer

Use the analysis results to extract a vector from the best-scoring layer.

In [ ]:
from psyctl.core.steering_extractor import SteeringExtractor
from psyctl.core.steering_applier import SteeringApplier

best_layer = results["top_k_layers"][0]
print(f"Best layer: {best_layer}")

extractor = SteeringExtractor()
output_path = Path("./vectors/best_layer.safetensors")

vectors = extractor.extract_steering_vector(
    model_name=MODEL_NAME,
    layers=[best_layer],
    dataset_path=dataset_path,
    output_path=output_path,
    method="mean_diff",
    normalize=True,
)

# Quick test
applier = SteeringApplier()
result = applier.apply_steering(
    model_name=MODEL_NAME,
    steering_vector_path=output_path,
    input_text="Hello, how are you?",
    strength=1.5,
    max_new_tokens=100,
    temperature=0.7,
)
print(f"\nSteered response (strength=1.5): {result}")